## Ensemble Model: BERT Embeddings + XGBoost

### 📋 Cel notebooka
Ten notebook implementuje model hybrydowy (Ensemble), który łączy:
1.  **BERT Embeddings:** Głębokie reprezentacje semantyczne z fine-tunowanego modelu BERT.
2.  **XGBoost:** Gradient boosting na połączonych cechach (embeddingi + klasyczne cechy binarne).

### 🧠 Dlaczego Ensemble?
Analiza wykazała, że modele drzewiaste (XGBoost, RF) radzą sobie lepiej z binarną naturą danych (obecność tokenów), podczas gdy BERT lepiej rozumie rzadsze relacje. Połączenie ich (użycie BERTa jako feature extractora) często daje najlepsze wyniki.

### 🔄 Wymagania
- `bert_triage_model_*` - wytrenowany model BERT (z notebooka 05)
- `corpus_train_*.txt` i `corpus_val_*.txt` - dane z podziału 05
- Biblioteki: `xgboost`, `transformers`, `torch`


In [ ]:
import torch
import numpy as np
from tqdm.auto import tqdm
from transformers import BertModel, PreTrainedTokenizerFast
from xgboost import XGBClassifier
from sklearn.metrics import classification_report, accuracy_score, f1_score
from sklearn.preprocessing import MultiLabelBinarizer
import glob
from pathlib import Path
import sys
import os

# Konfiguracja
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
BATCH_SIZE = 128  # Większy batch dla szybszej inferencji
print(f"🚀 Device: {DEVICE}")

# 1. Znajdź najnowsze pliki
triage_models = sorted([d for d in glob.glob("bert_triage_model_*") if Path(d).is_dir()], reverse=True)
if not triage_models:
    raise FileNotFoundError("Brak modelu BERT! Uruchom najpierw 05_Fine_Tune_Model.ipynb")
MODEL_PATH = triage_models[0]

# Check for checkpoints if main dir is empty of model files
if not (Path(MODEL_PATH) / "config.json").exists():
    checkpoints = sorted([d for d in glob.glob(f"{MODEL_PATH}/checkpoint-*") if Path(d).is_dir()], key=lambda x: int(x.split('-')[-1]), reverse=True)
    if checkpoints:
        MODEL_PATH = checkpoints[0]
        print(f"⚠️ Używam checkpointu: {MODEL_PATH}")
    else:
        print(f"⚠️ UWAGA: Katalog {MODEL_PATH} nie zawiera modelu ani checkpointów!")

print(f"📂 Model BERT: {MODEL_PATH}")

val_files = sorted(glob.glob("corpus_val_*.txt"), reverse=True)
train_files = sorted(glob.glob("corpus_train_*.txt"), reverse=True)

if not val_files or not train_files:
    raise FileNotFoundError("Brak plików corpus_*.txt! Uruchom najpierw 05.")

TRAIN_FILE = train_files[0]
VAL_FILE = val_files[0]
print(f"📂 Train Data: {TRAIN_FILE}")
print(f"📂 Val Data: {VAL_FILE}")

# 2. Import label mapping
current_dir = Path(os.getcwd())
project_root = None
# Szukaj label_mapping.py w katalogu nadrzędnym lub bieżącym
for path in [current_dir, current_dir.parent]:
    if (path / "label_mapping.py").exists():
        project_root = path
        break

if project_root:
    sys.path.insert(0, str(project_root))
    from label_mapping import LABEL_TO_ID, LABEL_NAMES
    print(f"✅ Załadowano mapowanie etykiet z {project_root}")
else:
    raise FileNotFoundError("Nie znaleziono label_mapping.py!")

In [ ]:
# 3. Ładowanie modelu BERT (jako Feature Extractor)
print("⏳ Ładowanie modelu BERT...")

try:
    # Najpierw spróbujmy załadować z MODEL_PATH jeśli tam jest tokenizer.json
    if (Path(MODEL_PATH) / "tokenizer.json").exists():
        tokenizer = PreTrainedTokenizerFast(tokenizer_file=str(Path(MODEL_PATH) / "tokenizer.json"))
    else:
        # Fallback do MLM
        mlm_models = sorted([d for d in glob.glob("medical_bert_mlm_*") if Path(d).is_dir()], reverse=True)
        if mlm_models and (Path(mlm_models[0]) / "tokenizer.json").exists():
            print(f"⚠️ Używam tokenizera z MLM: {mlm_models[0]}")
            tokenizer = PreTrainedTokenizerFast(tokenizer_file=str(Path(mlm_models[0]) / "tokenizer.json"))
        else:
            # Ostateczny fallback - może biblioteka sama znajdzie
            tokenizer = PreTrainedTokenizerFast.from_pretrained(MODEL_PATH)

    # Konfiguracja tokenów specjalnych (ważne dla fast tokenizer)
    tokenizer.pad_token = "[PAD]"
    tokenizer.cls_token = "[CLS]"
    tokenizer.sep_token = "[SEP]"
    tokenizer.mask_token = "[MASK]"
    tokenizer.unk_token = "[UNK]"
    
except Exception as e:
    print(f"❌ BŁĄD ładowania tokenizera: {e}")
    raise

# Ładujemy tylko base model (bez głowicy klasyfikacyjnej), aby wyciągnąć embeddingi
bert_model = BertModel.from_pretrained(MODEL_PATH)
bert_model.to(DEVICE)
bert_model.eval()
print("✅ Model załadowany. Będzie używany do ekstrakcji embeddingów.")

In [ ]:
def load_data(filepath):
    sequences = []
    labels = []
    with open(filepath, 'r') as f:
        for line in f:
            line = line.strip()
            if line:
                parts = line.split()
                label_str = parts[-1]
                text = " ".join(parts[:-1])
                if label_str in LABEL_TO_ID:
                    sequences.append(text)
                    labels.append(LABEL_TO_ID[label_str])
    return sequences, labels

print("⏳ Wczytywanie danych...")
X_train_text, y_train = load_data(TRAIN_FILE)
X_val_text, y_val = load_data(VAL_FILE)
print(f"✅ Train: {len(X_train_text)}")
print(f"✅ Val: {len(X_val_text)}")

In [ ]:
def extract_bert_embeddings(texts, batch_size=BATCH_SIZE):
    embeddings = []
    
    # Batch processing
    for i in tqdm(range(0, len(texts), batch_size), desc="Ekstrakcja embeddingów"):
        batch_texts = texts[i:i+batch_size]
        
        # Tokenizacja
        inputs = tokenizer(
            batch_texts,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=128
        ).to(DEVICE)
        
        with torch.no_grad():
            outputs = bert_model(**inputs)
            # Bierzemy wektor [CLS] (pierwszy token)
            cls_embeddings = outputs.last_hidden_state[:, 0, :]
            embeddings.append(cls_embeddings.cpu().numpy())
            
    return np.vstack(embeddings)

print("🚀 Generowanie embeddingów dla zbioru treningowego...")
X_train_emb = extract_bert_embeddings(X_train_text)

print("🚀 Generowanie embeddingów dla zbioru walidacyjnego...")
X_val_emb = extract_bert_embeddings(X_val_text)

print(f"✅ Kształt embeddingów Train: {X_train_emb.shape}")

In [ ]:
print("🔍 Generowanie klasycznych cech (Bag of Words) dla XGBoost...")

# Prosty tokenizer: split po spacji
def simple_tokenize(text):
    return text.split()

# Używamy MultiLabelBinarizer do stworzenia one-hot/multi-hot encodingu tokenów
# To jest to samo co robił Random Forest w baseline
mlb = MultiLabelBinarizer(sparse_output=False) # XGBoost woli dense numpy arrays zazwyczaj, ale memory usage może być issue
# Dla małego słownika (139 tokenów) dense jest OK.

# Fit na train
train_tokens = [simple_tokenize(t) for t in X_train_text]
val_tokens = [simple_tokenize(t) for t in X_val_text]

X_train_bow = mlb.fit_transform(train_tokens)
X_val_bow = mlb.transform(val_tokens)

print(f"✅ Vocab size: {len(mlb.classes_)}")
print(f"✅ Kształt BOW Train: {X_train_bow.shape}")

In [ ]:
print("🔗 Łączenie cech (Embeddings + BOW)...")
X_train_combined = np.hstack([X_train_emb, X_train_bow])
X_val_combined = np.hstack([X_val_emb, X_val_bow])

print(f"✅ Finalny kształt Train: {X_train_combined.shape}")

In [ ]:
print("🏋️ Trening XGBoost...")
xgb = XGBClassifier(
    n_estimators=200,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    objective='multi:softmax',
    num_class=len(LABEL_NAMES),
    random_state=42,
    n_jobs=-1,
    tree_method='hist', # Szybsze dla dużej ilości danych
    device='cuda' if torch.cuda.is_available() else 'cpu'
)

xgb.fit(X_train_combined, y_train, eval_set=[(X_val_combined, y_val)], verbose=10)

print("✅ Trening zakończony!")

In [ ]:
print("📊 Ewaluacja...")
y_pred = xgb.predict(X_val_combined)

acc = accuracy_score(y_val, y_pred)
f1 = f1_score(y_val, y_pred, average='weighted')

print("\n🏆 XGBoost + BERT Embeddings Results:")
print(f"   Accuracy: {acc:.4f}")
print(f"   F1 Score: {f1:.4f}")

print("\n🔍 Raport klasyfikacji:")
print(classification_report(y_val, y_pred, target_names=LABEL_NAMES))

# Porównanie z samym BERT (z logów lub poprzedniego nb)
print("\n⚖️ Porównanie (Baseline vs Ensemble):")
print("   BERT Only:      ~0.655 Acc")
print("   Random Forest:  ~0.705 Acc")
print(f"   Ensemble:       {acc:.4f} Acc")